# 03 · Stream a CommunityForensics-Small subset (unseen generators)

**Accelerator:** none · **Runtime:** 20–60 min (network bound) · **Output dataset name:** `cf-subset`

We never download the 260 GB dataset. The loader streams `OwensLab/CommunityForensics-Small` (split `train`,
bytes in `image_data`, generator in `model_name`, label 1 = fake; verified against the dataset card) and keeps
explicit quotas: 1500 real + 1500 fake, at most 40 fakes per model, bounded scan. Re-running resumes from the
manifest. Licence: CC BY-NC-SA 4.0 (each generated image also carries its source model's licence).

## 0. Setup (same cell in every notebook)

**Before running:** attach the code as a Kaggle Dataset named `listing-verifier-code`
(upload the repo folder without `data/`, `outputs/`, `web/node_modules/`), **or** set `REPO_URL`
to your GitHub repo. Turn on **Internet** (pip, ABO and HF downloads). For the GPU notebooks pick
**Settings → Accelerator → GPU T4 x2**. Store your Hugging Face token as the Kaggle secret `HF_TOKEN`.

In [ ]:
import os, shutil, subprocess, sys, pathlib, json
REPO_URL = "https://github.com/kvamsi-iisc/DLVA-Final-Project-Forensic-Verification-of-Marketplace-Listings.git"   # used if the code dataset is not attached
WORK = pathlib.Path("/kaggle/working"); REPO = WORK / "repo"; DATA = WORK / "data"; OUT = WORK / "outputs"
code_ds = pathlib.Path("/kaggle/input/listing-verifier-code")
if not REPO.exists():
    if code_ds.exists():
        shutil.copytree(code_ds, REPO)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
os.environ["LV_DATA_ROOT"] = str(DATA); os.environ["LV_OUTPUT_ROOT"] = str(OUT)
os.environ["HF_HOME"] = str(WORK / "hf_cache")           # model weights land here (removed before publishing)
DATA.mkdir(exist_ok=True); OUT.mkdir(exist_ok=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("no HF_TOKEN secret:", type(e).__name__, "(fine unless a gated model is needed)")
print("python", sys.version.split()[0])
pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], capture_output=True, text=True)
if pip.returncode != 0:
    print(pip.stdout[-3000:], pip.stderr[-3000:])          # show WHY before stopping (e.g. a pin without a wheel for this Python)
    raise SystemExit("pip install failed; see output above")
subprocess.run([sys.executable, "-m", "pip", "check"], check=False)

LOGS = OUT / "logs"; LOGS.mkdir(parents=True, exist_ok=True)

def run(*args):
    """Run a pipeline script; stream its output to the cell AND to OUT/logs/<script>_<time>.log,
    append a completion record to OUT/logs/completed.jsonl, and stop the notebook on a non-zero exit."""
    import time, datetime
    cmd = [sys.executable, *[str(a) for a in args]]
    name = pathlib.Path(str(args[0])).stem
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    log_path = LOGS / f"{name}_{stamp}.log"
    print(">>>", " ".join(cmd[1:]), "->", log_path.name, flush=True)
    t0 = time.time()
    with open(log_path, "w") as log:
        log.write("$ " + " ".join(cmd) + "\n")
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end="", flush=True); log.write(line)
        rc = proc.wait()
    record = {"script": name, "args": cmd[2:], "returncode": rc, "seconds": round(time.time() - t0, 1),
              "log": str(log_path), "finished_utc": datetime.datetime.now(datetime.timezone.utc).isoformat()}
    with open(LOGS / "completed.jsonl", "a") as f:
        f.write(json.dumps(record) + "\n")
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

def attach(slug, sub, dest, must_contain=None):
    """Copy a sub-folder of an attached Kaggle dataset into DATA/OUT (copy, not symlink: scripts write next to inputs).
    Refuses silently stale copies: if `must_contain` is missing from an existing dest, the copy is redone."""
    base = pathlib.Path("/kaggle/input") / slug
    src = base / sub if (base / sub).exists() else base
    dst = pathlib.Path(dest)
    if dst.exists() and (must_contain is None or (dst / must_contain).exists()):
        print("exists:", dst); return
    if dst.exists():
        shutil.rmtree(dst)
    if not src.exists():
        raise FileNotFoundError(f"dataset {slug} is not attached (expected {src})")
    print("copy", src, "->", dst); shutil.copytree(src, dst)

run("scripts/env_report.py", "--out", OUT / "results" / "env_report.json")
import torch; print(f"visible GPUs: {torch.cuda.device_count()}")


In [ ]:
run("scripts/04_stream_forensics.py", "--inspect")

If the printed feature names differ from `image_data` / `label` / `model_name`, pass `--image-col/--label-col/--generator-col`. Then verify by eye that label 1 is fake.

In [ ]:
run("scripts/04_stream_forensics.py", "--n-real", 1500, "--n-fake", 1500, "--per-generator", 40, "--max-scan", 200000)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from PIL import Image
df = pd.read_csv(DATA / "forensics" / "forensics_subset.csv"); print(df["label"].value_counts()); print(df["generator"].nunique(), "generators;", df["architecture"].value_counts().head(10) if "architecture" in df else "")
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for ax, (_, r) in zip(axes.ravel(), df.groupby("label").head(6).iterrows()):
    ax.imshow(Image.open(r["path"])); ax.set_title(f"label={r['label']} {str(r['generator'])[:18]}", fontsize=8); ax.axis("off")
plt.tight_layout()

## Persisting outputs
When this notebook finishes under **Save Version → Save & Run All (Commit)**, everything in `/kaggle/working`
except the model cache is kept as the notebook output. Then: *Output tab → "New Dataset"* and name it as stated
at the top of this notebook. The next notebook attaches that dataset. The cell below checks the artifacts this
notebook promised actually exist, then removes the model cache so the output stays small.

In [ ]:
required = ["data/forensics/forensics_subset.csv"]
run("scripts/verify_outputs.py", *[WORK / r for r in required])   # exits non-zero on a missing/empty artifact
shutil.rmtree(WORK / "hf_cache", ignore_errors=True)
shutil.rmtree(REPO / "web" / "node_modules", ignore_errors=True)
print(subprocess.run(["du", "-sh", str(DATA), str(OUT)], capture_output=True, text=True).stdout)